# RAG Pipeline Exploration

This notebook explores the RAG pipeline from ingestion to generation for a single PDF.
We will:
1. Parse a PDF using the current `Docling` parser.
2. Inspect extracted text, images, and bounding boxes (for deep linking).
3. Chunk the text.
4. Generate embeddings.
5. Perform retrieval.
6. Generate an answer.

## Setup
Ensure you have the environment variables set (OpenAI API Key).

In [ ]:
import os
import sys
import logging
from pathlib import Path

# Add project root to path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. Document Parsing (Docling)

We use `rag.ingest.parsers.docling_parser` to parse the PDF. 
Key features we are checking:
- **Image Extraction**: Are images saved to `data/assets`?
- **Bounding Boxes**: Do we get `bbox` coordinates for deep linking?

In [ ]:
from rag.ingest.parsers.docling_parser import parse_pdf_multimodal

# Select a sample PDF
pdf_path = "../data/raw/00294fcd04e8_NLP2103_QUT_Final_Report_%283%29.pdf"  # Change this to your PDF
output_dir = "../data/assets"

print(f"Parsing {pdf_path}...")
parsed_doc = parse_pdf_multimodal(
    pdf_path=pdf_path,
    output_dir=output_dir,
    extract_images=True,
    extract_tables=True
)

print(f"Parsed {len(parsed_doc.elements)} elements.")
print(f"Metadata: {parsed_doc.metadata}")

### Inspect Elements
Let's look at the first few elements to see if `bbox` and `image_path` are present.

In [ ]:
for i, elem in enumerate(parsed_doc.elements[:10]):
    print(f"[{i}] Type: {elem.type}, Page: {elem.page}")
    print(f"    BBox: {elem.bbox}")
    if elem.type == "image":
        print(f"    Image Path: {elem.image_path}")
    print(f"    Text: {elem.text[:50]}...")
    print("-" * 40)

## 2. Chunking
We use semantic chunking to split the text into meaningful blocks.

In [ ]:
from rag.ingest.chunkers.semantic import SemanticChunker

chunker = SemanticChunker()
chunks = chunker.chunk(parsed_doc.full_text)

print(f"Generated {len(chunks)} chunks.")
print(f"Sample Chunk: {chunks[0].text[:100]}...")

## 3. Embedding & Retrieval
We'll use OpenAI embeddings and a simple in-memory FAISS index for this demo.

In [ ]:
import numpy as np
import faiss
from app.adapters.loader import load_embedder

# Load embedder (OpenAI)
embedder = load_embedder("openai")

# Embed chunks
texts = [c.text for c in chunks]
embeddings = embedder.embed_documents(texts)
embeddings_np = np.array(embeddings).astype("float32")

# Build Index
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_np)

print(f"Indexed {index.ntotal} vectors of dim {dimension}.")

## 4. Generation (RAG)
Ask a question and generate an answer.

In [ ]:
from app.adapters.llm_openai import OpenAIAdapter

question = "What are the key findings of this report?"

# 1. Embed Question
q_embed = np.array(embedder.embed_query(question)).astype("float32").reshape(1, -1)

# 2. Retrieve
k = 5
D, I = index.search(q_embed, k)

# 3. Prepare Context
retrieved_chunks = [chunks[i] for i in I[0]]
context = "\n\n".join([f"[Source {i+1}] {c.text}" for i, c in enumerate(retrieved_chunks)])

# 4. Generate
llm = OpenAIAdapter(model="gpt-4o")
prompt = f"Question: {question}\n\nContext:\n{context}\n\nAnswer:"

answer, usage = llm.chat([{"role": "user", "content": prompt}])

print("Answer:")
print(answer)

## LlamaParse Analysis

**What is LlamaParse?**
- A proprietary parsing service by LlamaIndex.
- Specialized in complex layouts (tables, columns, forms).
- Outputs Markdown.

**Cost:**
- **Free Tier:** 1,000 pages per day (generous).
- **Paid:** ~$0.003 per page (after free limit).

**Comparison:**
- **Docling (Current):** Open-source, runs locally. Good for general text, but might struggle with complex tables or specific bbox extraction if PDF metadata is messy.
- **LlamaParse:** Cloud-based, slower (network), but SOTA for layout preservation.

**To use LlamaParse:**
1. Get API Key from `cloud.llamaindex.ai`.
2. `pip install llama-parse`.
3. Replace `parse_pdf_multimodal` with LlamaParse call.